# Easy Q3 — 2020 年夏天台北市區 vs 陽明山地表溫差

## 資料集選擇說明

選用 **MODIS Terra LST (`MODIS/061/MOD11A1`)** 每日白天 1 km 地表溫度：

| 屬性 | 說明 |
|------|------|
| 時間解析度 | 每日（每日合成）|
| 空間解析度 | 1 km |
| 原始單位 | 0.02 K（需 × 0.02 − 273.15 → °C）|
| 可用時間範圍 | 2000-02-24 至今 |
| 品質篩選 | 使用 `QC_Day` band bit 0–1 == 0（good quality）|

空間解析度 1 km 適合分辨台北盆地（低地）與陽明山（山地）的溫差。

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

ee.Authenticate()
ee.Initialize(project='0')

In [5]:
# 研究點位
taipei_city_pt  = ee.Geometry.Point([121.5654, 25.0330])   # 台北市政府附近
yangmingshan_pt = ee.Geometry.Point([121.5609, 25.1572])   # 陽明山主峰附近

# 10 km 圓形 buffer
taipei_buf  = taipei_city_pt.buffer(30000)
yangming_buf = yangmingshan_pt.buffer(30000)

START = '2020-08-30'
END   = '2020-08-31'

In [6]:
# 品質篩選函數：bit 0-1 == 0 (good LST)
def mask_lst_quality(image):
    qc = image.select('QC_Day')
    good = qc.bitwiseAnd(3).eq(0)  # bit 0-1 = 00
    return image.updateMask(good)

# 讀取 MODIS MOD11A1 白天 LST
lst_col = (ee.ImageCollection('MODIS/061/MOD11A1')
           .filterDate(START, END)
           .select(['LST_Day_1km', 'QC_Day'])
           .map(mask_lst_quality))

print('影像數:', lst_col.size().getInfo())

影像數: 1


In [7]:
# 換算攝氏度
def to_celsius(image):
    return image.select('LST_Day_1km') \
        .multiply(0.02).subtract(273.15) \
        .rename('LST_C') \
        .copyProperties(image, ['system:time_start'])

lst_c = lst_col.map(to_celsius)

# 兩個月平均地表溫度圖
lst_mean = lst_c.mean()

In [ ]:
# 使用 geemap 顯示平均 LST 地圖
aoi = taipei_buf.union(yangming_buf)

Map = geemap.Map(center=[25.1, 121.55], zoom=11)
vis_params = {
    'min': 20, 'max': 45,
    'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
}
Map.addLayer(lst_mean.clip(aoi), vis_params, '2020 Jul-Aug Mean LST (°C)')
Map.addLayer(ee.Image().paint(taipei_buf, 1, 2), {'palette': 'white'}, '台北市區 buffer')
Map.addLayer(ee.Image().paint(yangming_buf, 1, 2), {'palette': 'lime'}, '陽明山 buffer')
Map.add_colorbar(vis_params, label='LST (°C)')
Map

In [ ]:
# 計算兩 buffer 內的平均 LST（整段期間平均）
def mean_in_region(image, region, name):
    val = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=1000,
        maxPixels=1e8
    ).get('LST_C')
    return val

taipei_mean   = mean_in_region(lst_mean, taipei_buf, '台北市區')
yangming_mean = mean_in_region(lst_mean, yangming_buf, '陽明山')

t_val = taipei_mean.getInfo()
y_val = yangming_mean.getInfo()
diff  = t_val - y_val

summary = pd.DataFrame({
    '地點': ['台北市區', '陽明山'],
    '2020年7-8月平均地表溫度 (°C)': [round(t_val, 2), round(y_val, 2)]
})
summary.loc[2] = ['溫差 (市區 − 陽明山)', round(diff, 2)]
print(summary.to_string(index=False))

None


TypeError: unsupported operand type(s) for -: 'NoneType' and 'NoneType'

In [ ]:
# 每日平均 LST 時間序列圖
def region_mean_timeseries(collection, region):
    def extract(image):
        mean = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=1000,
            maxPixels=1e8
        )
        return ee.Feature(None, {
            'date': image.date().format('YYYY-MM-dd'),
            'LST_C': mean.get('LST_C')
        })
    return ee.FeatureCollection(collection.map(extract))

fc_taipei   = region_mean_timeseries(lst_c, taipei_buf)
fc_yangming = region_mean_timeseries(lst_c, yangming_buf)

def fc_to_df(fc):
    rows = fc.getInfo()['features']
    return pd.DataFrame([r['properties'] for r in rows]).dropna()

df_taipei   = fc_to_df(fc_taipei)
df_yangming = fc_to_df(fc_yangming)
df_taipei['date']   = pd.to_datetime(df_taipei['date'])
df_yangming['date'] = pd.to_datetime(df_yangming['date'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_taipei['date'],   df_taipei['LST_C'],   label='台北市區',  color='red')
ax.plot(df_yangming['date'], df_yangming['LST_C'], label='陽明山', color='green')
ax.set_xlabel('日期')
ax.set_ylabel('地表溫度 (°C)')
ax.set_title('2020 年 7–8 月白天地表溫度（MODIS MOD11A1）')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('lst_timeseries_2020.png', dpi=150)
plt.show()
print('圖表已存為 lst_timeseries_2020.png')